# 06 — Analyse d'erreurs et recommandations

**Projet** : Espace vectoriel creux d'un catalogue dupliqué (TF-IDF cosinus) — la référence à battre

## Objectifs pédagogiques

1. Lire le verdict contractuel de l'évaluateur de production sur le split de test.
1. Ventiler les résultats par difficulté et par intention : trois segments, trois correctifs différents.
1. Mesurer le comportement d'abstention, puis proposer des actions reliées à un chiffre.

**Objectifs transverses du dépôt**

- Construire un index vectoriel déterministe et hors ligne : hachage de n-grammes, SVD tronquée apprise sur le train, normalisation L2.
- Mesurer les propriétés d'un index au lieu de les supposer : dimension réellement produite, fidélité de la projection, taille de l'artefact.
- Chiffrer ce que la compression apporte et ce qu'elle coûte : 128 dimensions contre 4096, 0,16 Mo de matrice contre 5,25 Mo, fidélité de projection de 1,00.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (64 documents) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 64

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

CHUNKING = dict(node(CONFIG, "preprocessing").get("chunking", {}))

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (plancher contractuel : 0.9)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(
    f"Découpage         : {CHUNKING.get('max_tokens')} tokens, "
    f"chevauchement {CHUNKING.get('overlap_tokens')}"
)
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

## 1. Le verdict

L'évaluation de production écrit un rapport et un fichier de métriques ; on les relit ici sans les recalculer, pour qu'un notebook ne puisse jamais publier un chiffre différent de celui du rapport. C'est cette même chaîne qui tourne en CI : l'évaluation charge l'artefact entraîné, mesure le split de test une seule fois et compare la métrique principale au plancher trivial.

In [ ]:
from src.data.generators import SyntheticCorpusGenerator
from src.data.loaders import TextCorpusLoader

LOADER = TextCorpusLoader(PATHS, formats=CONFIG.data.formats)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    QUERIES = LOADER.load_queries()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} documents, {len(QUERIES)} questions")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, QUERIES, METADATA = bundle.documents, bundle.queries, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_queries(QUERIES)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} documents généré et persisté")

CALIBRATION = QUERIES[QUERIES["split"] == "calibration"].reset_index(drop=True)
VALIDATION = QUERIES[QUERIES["split"] == "val"].reset_index(drop=True)
TEST = QUERIES[QUERIES["split"] == "test"].reset_index(drop=True)
ANSWERABLE = QUERIES[QUERIES["answer_type"] != "unanswerable"].reset_index(drop=True)
# Sous-ensemble utilisé pour **régler** quoi que ce soit : la validation seule. Le test ne sert
# qu'à juger, jamais à choisir (y compris dans un notebook).
VAL_ANSWERABLE = VALIDATION[VALIDATION["answer_type"] != "unanswerable"].reset_index(drop=True)
print(
    "splits :",
    {
        name: len(frame)
        for name, frame in (("calibration", CALIBRATION), ("val", VALIDATION), ("test", TEST))
    },
)

In [ ]:
from src.models import build_model, load_model
from src.pipelines.evaluation_pipeline import EvaluationPipeline

ARTIFACT = NB_PATHS.models_dir / "lexical_index.joblib"
if not ARTIFACT.exists():
    # Notebook exécuté seul : on construit un index de travail dans le bac à sable plutôt que de
    # lire `artifacts/models/`, qui appartient à `make train`.
    print("artefact absent de outputs/notebooks : construction d'un index de travail")
    bootstrap = build_model(CONFIG.model_dump())
    bootstrap.fit(DOCUMENTS, CALIBRATION)
    bootstrap.save(ARTIFACT)

TRAINED_MODEL = load_model(ARTIFACT, config=CONFIG.model_dump())
EVALUATION = EvaluationPipeline(CONFIG, paths=NB_PATHS).run()
print("statut :", EVALUATION.status)
for message in EVALUATION.messages:
    print("-", message)

In [ ]:
from IPython.display import Markdown

REPORT_PATH = NB_PATHS.reports_dir / "evaluation_report.md"
display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))

## 2. Résultats par segment

Un rappel global peut cacher un segment excellent et un autre en échec : ce sont les écarts qui dictent le travail suivant.

In [ ]:
RESULT = EVALUATION.payload
segments = pd.DataFrame(RESULT.segments).T.astype(float)
columns = [
    name
    for name in ("n_questions", "recall_at_5", "mrr", "abstention_rate")
    if name in segments.columns
]
segments[columns].round(3).sort_values("recall_at_5")

In [ ]:
per_question = RESULT.per_question.copy()
per_question["trouvé"] = per_question["first_relevant_rank"] > 0
answerable_rows = per_question[per_question["answer_type"] != "unanswerable"]
by_difficulty = (
    answerable_rows.groupby("difficulty")
    .agg(
        questions=("query_id", "size"),
        rappel_moyen=("recall_at_5", "mean"),
        rang_moyen=("first_relevant_rank", lambda values: values[values > 0].mean()),
        non_trouvées=("trouvé", lambda values: int((~values).sum())),
    )
    .round(3)
)
by_difficulty

### 2.1 Les questions perdues, une par une

Un taux d'échec n'est pas une explication. On regarde les questions concernées : partagent-elles une formulation, un thème, une source ?

In [ ]:
lost = answerable_rows[~answerable_rows["trouvé"]].sort_values("top_score", ascending=False)
print(
    f"{len(lost)} questions sans passage pertinent dans le top-5 "
    f"sur {len(answerable_rows)} questions répondables"
)
display(lost[["question", "difficulty", "intent", "n_gold_docs", "top_score"]].head(10))

In [ ]:
# Ce que le système répond sur une question perdue : cela distingue « mauvais classement » (le bon
# passage existe mais arrive trop loin) de « réponse construite de toutes pièces ».
if len(lost):
    record = lost.iloc[0]
    answer = TRAINED_MODEL.answer(str(record["question"]), 5, query_id=str(record["query_id"]))
    print("question       :", record["question"])
    print("réponse        :", answer.text or "(abstention)")
    print("meilleur score :", round(answer.top_score, 3))
    display(pd.DataFrame([passage.to_row() for passage in answer.passages]))

## 3. Le comportement d'abstention

C'est la mesure que l'on oublie : un assistant qui répond toujours obtient de bons scores de rappel et invente des réponses sur les questions hors corpus.

In [ ]:
out_of_corpus = per_question[per_question["answer_type"] == "unanswerable"]
refusal_rate = float(out_of_corpus["abstained"].mean()) if len(out_of_corpus) else float("nan")
coverage = 1.0 - float(answerable_rows["abstained"].mean())
print(f"questions hors corpus            : {len(out_of_corpus)}")
print(f"refus corrects (taux)            : {refusal_rate:.1%}")
print(f"couverture des questions servies : {coverage:.1%}")
print(f"fausses réponses restantes       : {int((out_of_corpus['abstained'] == 0).sum())}")

In [ ]:
from IPython.display import Image

figures = sorted(NB_PATHS.figures_dir.glob("*.png"))
print("figures produites par l'évaluation :", [figure.name for figure in figures])
for figure in figures[:2]:
    display(Image(filename=str(figure)))

## 4. Recommandations

Chaque recommandation est reliée à une mesure produite plus haut : ce qui n'est pas mesuré ici doit être mesuré avant d'être corrigé.

In [ ]:
# On classe les difficultés **répondables** : le segment hors corpus n'appelle pas un meilleur
# modèle mais une abstention, qui est mesurée juste au-dessus.
difficulty = {
    key.split("=", 1)[1]: row
    for key, row in segments.iterrows()
    if key.startswith("difficulty=") and key != "difficulty=hors_corpus"
}
worst = min(difficulty, key=lambda name: difficulty[name]["recall_at_5"])
recommendations = [
    {
        "action": f"Traiter en priorité le segment « {worst} »",
        "preuve": (
            f"rappel@5 = {difficulty[worst]['recall_at_5']:.3f} sur "
            f"{int(difficulty[worst]['n_questions'])} questions"
        ),
        "effort": "moyen (modèle plus riche, latence supérieure)",
    },
    {
        "action": "Arbitrer le seuil de refus avec le métier",
        "preuve": f"{refusal_rate:.1%} de refus corrects pour {coverage:.1%} de couverture",
        "effort": "faible (décision, pas de code)",
    },
    {
        "action": "Enrichir le corpus là où les questions sont perdues",
        "preuve": f"{len(lost)} questions sans passage pertinent dans le top-5",
        "effort": "élevé (travail éditorial)",
    },
]
pd.DataFrame(recommendations)

**Ce qu'il faut retenir**

- Le verdict contractuel se lit avec son référentiel : un rappel@5 ne vaut que comparé au plancher trivial mesuré sur le même corpus.
- Facile, paraphrase et multi-document appellent trois correctifs différents : un réglage global les dégraderait tous les trois.
- L'abstention est une fonctionnalité mesurée, pas un aveu d'échec : refuser une question hors corpus est le comportement attendu d'un assistant documentaire.